# Sales Forecast API: Machine Learning Modeling

## 1. Objective

The objective of this notebook is to train, evaluate and compare machine learning models for the Rossmann Store Sales prediction problem.

The dataset has already been cleaned and transformed during the previous notebooks. Therefore, this notebook focuses exclusively on the modeling stage.

The workflow consists of:

- Loading the processed datasets;
- Creating a chronological train-test split;
- Building a preprocessing pipeline;
- Training baseline and machine learning models;
- Comparing their performance;
- Saving the final pipeline for deployment through the API.

By the end of this notebook, we will have a production-ready Scikit-Learn pipeline that can be loaded directly by the FastAPI application.

## 2. Load processed data

### 2.1 Import libraries

In [ ]:
import pandas as pd

from common import (
    PROCESSED_DATA_DIR, 
    MODEL_DIR)

from ml_utils import (
    split_feature_types,
    chronological_split, 
    evaluate_regression, 
    print_metrics,
    compare_models,
    get_feature_importance,
    plot_feature_importance)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

import joblib

### 2.2 Load processed datasets

In [ ]:
X = pd.read_parquet(PROCESSED_DATA_DIR / "X.parquet")

y = pd.read_parquet(PROCESSED_DATA_DIR / "y.parquet").squeeze()

### 2.3 Inspect the data

Before building any model, it is good practice to verify that the processed datasets were loaded correctly.

Although the data has already been validated during the feature engineering stage, checking the dimensions, data types and a few sample rows helps detect accidental problems before training.

In [ ]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
display(X.head())

display(y.head())

In [ ]:
X.info()

In [ ]:
print(f"Target variable: {y.name}")
print(f"Missing values in X: {X.isna().sum().sum()}")
print(f"Missing values in y: {y.isna().sum()}")

## 3. Train-test split

Machine learning models should be evaluated on data that was not available during training.

Since this project uses historical sales data, the dataset is split chronologically instead of randomly. This better reflects the real forecasting scenario, where future observations are predicted using only past information.

### 3.1 Create a chronological split

In [ ]:
X_train, X_test, y_train, y_test = chronological_split("2015-06-01", X, y)

### 3.2 Verify the split

Before training the models, we verify that the split produced the expected time intervals and dataset sizes.

In [ ]:
print("Training period:")
print(f"{X_train['Date'].min().date()} -> {X_train['Date'].max().date()}")

print()

print("Test period:")
print(f"{X_test['Date'].min().date()} -> {X_test['Date'].max().date()}")

In [ ]:
print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")

In [ ]:
print(f"Train: {len(X_train)/len(X):.1%}")
print(f"Test : {len(X_test)/len(X):.1%}")

### 3.3 Remove the Date column

The `Date` column was preserved only to create the chronological split.

Since the temporal information has already been transformed into calendar-related features during feature engineering, the original date is no longer needed and is removed before preprocessing.

In [ ]:
X_train = X_train.drop(columns="Date")
X_test = X_test.drop(columns="Date")

## 4. Build the preprocessing pipeline

Machine learning models usually require different preprocessing strategies depending on the feature type.

Instead of manually transforming the dataset before training, we build a preprocessing pipeline using Scikit-Learn. This approach keeps the workflow reproducible and ensures that the same transformations are applied during both training and inference.

### 4.1 Identify feature types

The first step is to separate numerical and categorical features. This allows different preprocessing strategies to be applied to each group while keeping the pipeline flexible if new features are added in the future.

In [ ]:
numerical_features, categorical_features = split_feature_types(X_train)

print("Categorical features:")
print(categorical_features)

print()

print("Numerical features:")
print(numerical_features)

### 4.2 Define preprocessing steps

The numerical features do not require additional preprocessing because missing values were handled during feature engineering and tree-based models do not require feature scaling.

Categorical variables are transformed using One-Hot Encoding so that they can be used by the machine learning models.

In [ ]:
numeric_transformer = "passthrough"

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        )
    ]
)

### 4.3 Build the ColumnTransformer

The `ColumnTransformer` combines all preprocessing steps into a single object.

Each transformer is applied only to its corresponding feature group, allowing the entire preprocessing workflow to be treated as a single component during training and inference.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features,
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features,
        ),
    ]
)

## 5. Establish a baseline

Before training more sophisticated models, it is useful to establish a baseline.

A baseline provides a simple reference point that any machine learning model should outperform. If a more complex model cannot achieve better performance than the baseline, it suggests that the modeling approach or feature set should be revisited.

In this project, the baseline model will always predict the average sales observed in the training set.

### 5.1 Train the baseline model

In [ ]:
baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

### 5.2 Evaluate the baseline

The baseline model is evaluated using the same metrics that will later be applied to the machine learning models. This ensures a fair comparison.

In [ ]:
baseline_predictions = baseline.predict(X_test)

baseline_metrics = evaluate_regression(y_test, baseline_predictions)

print_metrics(baseline_metrics)

### 5.3 Interpretation

The baseline establishes the minimum expected performance for this problem.

Any machine learning model should improve upon these metrics. If a more complex model performs similarly to the baseline, additional feature engineering, model selection or hyperparameter tuning may be required.

## 6. Random Forest

### 6.1 Why Random Forest?

Random Forest is an ensemble learning algorithm based on multiple decision trees.

Each tree is trained on a different subset of the data, and the final prediction is obtained by averaging the predictions of all trees. This approach generally improves predictive performance while reducing overfitting compared to a single decision tree.

Random Forest is widely used for tabular datasets because it can model nonlinear relationships, requires little preprocessing and is relatively robust to noisy data.

### 6.2 Build the pipeline

The Random Forest model is combined with the preprocessing pipeline built earlier.

Using a Scikit-Learn pipeline ensures that the same preprocessing steps are consistently applied during both training and inference.

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=50,
                max_depth=20,
                min_samples_leaf=5,
                n_jobs=2,
                random_state=42,
            ),
        ),
    ]
)

### 6.3 Train the model

The pipeline is trained using the training subset created by the chronological split.

In [ ]:
rf_pipeline.fit(X_train, y_train)

### 6.4 Evaluate performance

The trained model is evaluated using the same metrics adopted for the baseline.

Using identical evaluation metrics allows a fair comparison between the baseline and the machine learning models.

In [ ]:
rf_predictions = rf_pipeline.predict(X_test)

rf_metrics = evaluate_regression(y_test, rf_predictions)

print_metrics(rf_metrics)

### 6.5 Feature importance

Tree-based models provide an estimate of how much each feature contributed to the prediction process.

Although feature importance does not imply causality, it is a useful tool for understanding which variables the model relied on most when making predictions.

This analysis also serves as a sanity check: if the most important features are consistent with domain knowledge, it increases confidence that the model learned meaningful patterns from the available historical data.

In [ ]:
feature_importance = get_feature_importance(
    rf_pipeline
)

feature_importance.head(15)

In [ ]:
plot_feature_importance(
    feature_importance
)

### 6.6 Interpretation

The feature importance ranking shows that the model relies on a combination of store-specific characteristics, promotional information and calendar-related variables.

Competition-related features, such as the distance to the nearest competitor and when the competition started operating, are among the most influential variables. Promotional variables also contribute substantially to the predictions, indicating that promotions play an important role in explaining sales.

The store identifier is also one of the most important features. This indicates that the model learned persistent differences between individual stores over time, although the feature importance alone does not explain which underlying factors are responsible for those differences.

Overall, the ranking is consistent with the business problem and provides additional confidence that the model learned meaningful patterns from the available historical data.

## 7. XGBoost

### 7.1 Why XGBoost?

XGBoost (Extreme Gradient Boosting) is an optimized implementation of the Gradient Boosting algorithm designed for efficiency and predictive performance.

Instead of building independent trees, XGBoost trains trees sequentially, with each new tree focusing on correcting the errors made by the previous ones.

It also incorporates several optimizations, such as regularization, efficient tree construction and parallelized computations, making it one of the most widely used algorithms for supervised learning on tabular data.

In this project, XGBoost is evaluated alongside the Random Forest model to determine which algorithm provides the best predictive performance.

### 7.2 Build the pipeline

As with the Random Forest model, XGBoost is combined with the preprocessing pipeline.

This guarantees that exactly the same preprocessing steps are applied during training and future predictions, allowing the algorithms to be compared under identical conditions.

In [ ]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=2,
            ),
        ),
    ]
)

### 7.3 Train the model

The XGBoost pipeline is trained using the same chronological training set adopted throughout this notebook.

Using the same training data ensures that any performance differences between the models are due to the learning algorithms rather than differences in the data.

In [ ]:
xgb_pipeline.fit(X_train, y_train)

### 7.4 Evaluate performance

The XGBoost model is evaluated using the same metrics applied to both the baseline and the Random Forest model.

This allows an objective comparison of predictive performance across all evaluated models.

In [ ]:
xgb_predictions = xgb_pipeline.predict(X_test)

xgb_metrics = evaluate_regression(y_test, xgb_predictions)

print_metrics(xgb_metrics)

### 7.5 Interpretation

The XGBoost model substantially outperformed the baseline, confirming its ability to learn meaningful patterns from the historical data.

However, under the current feature engineering strategy and hyperparameter configuration, it did not achieve the same predictive performance as the Random Forest model.

The next section compares all evaluated models side by side to support the selection of the final production model.

## 8. Model comparison

### 8.1 Compare all models

The baseline and both machine learning models are compared using the same evaluation metrics.

Presenting all results together provides an objective basis for selecting the model that will be deployed in production.

In [ ]:
results = compare_models(
    {
        "Baseline": baseline_metrics,
        "Random Forest": rf_metrics,
        "XGBoost": xgb_metrics,
    }
)

results

### 8.2 Discussion

Both machine learning models substantially outperformed the baseline, demonstrating that the engineered features capture meaningful information about future sales.

Under the evaluated configuration, the Random Forest model consistently achieved the lowest prediction errors, indicating that it was better suited to this forecasting task than XGBoost.

These results also highlight an important aspect of machine learning: the most appropriate algorithm depends on the dataset, the available features and the chosen hyperparameters. A more sophisticated algorithm does not necessarily produce better predictions.

### 8.3 Select the final model

Among the evaluated models, the Random Forest achieved the best overall predictive performance on the validation period.

Since the objective of this project is to deploy the model with the best predictive performance, the Random Forest pipeline is selected as the production model.

It is important to note that this conclusion is specific to the current dataset, feature engineering strategy and hyperparameter configuration. Different settings could lead to different results.

## 9. Save the production pipeline

### 9.1 Save the selected model

The selected Random Forest pipeline is serialized using Joblib so that it can be loaded directly by the FastAPI application during inference.

Saving the complete Scikit-Learn pipeline preserves both the preprocessing steps and the trained model, ensuring that the exact same transformations applied during training are also applied when generating predictions for new data.

In [ ]:
joblib.dump(
    rf_pipeline,
    MODEL_DIR / "sales_forecast_pipeline.joblib",
)

## 10. Conclusion

In this notebook, a complete machine learning workflow was implemented for the Rossmann sales forecasting problem.

A chronological train-test split was adopted to simulate a real forecasting scenario, and all preprocessing steps were integrated into Scikit-Learn pipelines to ensure consistency between training and inference.

The evaluated models were compared using the same validation period, allowing an objective assessment of their predictive performance. Under the current experimental setup, the Random Forest model achieved the best overall results and was therefore selected for deployment.

Finally, the selected pipeline was serialized as a production artifact, making it ready to be loaded by the FastAPI application for real-time sales predictions.